# Age vs Music

How much do music habits add to predicting each mental-health target beyond age? Compares three
feature sets (age only, music only, age + music) on the same 50 outer folds as the nested CV, on
the training set only.

- Models: mean baseline, Ridge with a cubic spline for age (so the age-only model can fit a
  curved age effect), and the tuned random forest with raw age. Hyperparameters are tuned inside
  each outer fold, as in the nested CV.
- The main result is ΔR² = R²(age + music) − R²(age only), computed fold by fold with the
  corrected 95% CI. Music adds predictive information beyond age only if this CI is above zero.

In [ ]:
import pandas as pd

import mxmh_common as mc

# Set RERUN = False to use the saved per-fold results if they exist.
RERUN = True
FOLDS_PATH = mc.RESULTS_DIR / "age_vs_music_folds.csv"

if RERUN or not FOLDS_PATH.exists():
    folds = mc.run_age_vs_music()
else:
    folds = pd.read_csv(FOLDS_PATH)
    print(f"Loaded {len(folds)} fold results from {FOLDS_PATH}")

# Training and held-out sizes of each outer fold, needed for the corrected CI
X, Y = mc.load_clean_dataset()
X_train, _, _, _ = mc.split_train_test(X, Y)
n_train, n_test = mc.outer_fold_sizes(len(X_train))

## Performance by Feature Set

Mean across the 50 outer folds, with the corrected 95% CI for R², RMSE and MAE.

In [ ]:
MODEL_ORDER = ["Mean baseline", "Ridge", "RF tuned"]
SET_ORDER = ["No features"] + list(mc.FEATURE_SETS)


def fmt_ci(ci, digits):
    return f"{ci['mean']:.{digits}f} [{ci['ci_low']:.{digits}f}, {ci['ci_high']:.{digits}f}]"


rows = []
for (target, model, feature_set), group in folds.groupby(["target", "model", "feature_set"]):
    rows.append({"Target": target, "Model": model, "Feature set": feature_set,
                 "R2": fmt_ci(mc.corrected_ci(group.R2, n_train, n_test), 3),
                 "RMSE": fmt_ci(mc.corrected_ci(group.RMSE, n_train, n_test), 2),
                 "MAE": fmt_ci(mc.corrected_ci(group.MAE, n_train, n_test), 2)})
by_set = pd.DataFrame(rows)
by_set["Model"] = pd.Categorical(by_set.Model, MODEL_ORDER, ordered=True)
by_set["Feature set"] = pd.Categorical(by_set["Feature set"], SET_ORDER, ordered=True)
by_set = by_set.sort_values(["Target", "Model", "Feature set"])
by_set.to_csv(mc.RESULTS_DIR / "age_vs_music_by_set.csv", index=False)
by_set.set_index(["Target", "Model", "Feature set"])

## Increment from Adding Music to Age

Fold-by-fold difference between the age + music and age-only models, with the corrected 95% CI. Positive ΔR² and
negative ΔRMSE and ΔMAE mean adding music improves the predictions. "Music helps" is the share of
folds with ΔR² > 0.

In [ ]:
rows = []
for (target, model), group in folds[folds.model != "Mean baseline"].groupby(["target", "model"]):
    wide = group.pivot_table(index="fold", columns="feature_set", values=["R2", "RMSE", "MAE"])
    diff = {metric: mc.corrected_ci(wide[metric]["Age + music"] - wide[metric]["Age only"],
                                    n_train, n_test)
            for metric in ["R2", "RMSE", "MAE"]}
    rows.append({"Target": target, "Model": model,
                 "ΔR2": fmt_ci(diff["R2"], 3),
                 "p (corrected)": round(diff["R2"]["p_corrected"], 3),
                 "Music helps": f"{diff['R2']['share_positive']:.0%}",
                 "ΔRMSE": fmt_ci(diff["RMSE"], 3),
                 "ΔMAE": fmt_ci(diff["MAE"], 3)})
increment = pd.DataFrame(rows)
increment["Model"] = pd.Categorical(increment.Model, MODEL_ORDER, ordered=True)
increment = increment.sort_values(["Target", "Model"])
increment.to_csv(mc.RESULTS_DIR / "age_vs_music_increment.csv", index=False)
increment.set_index(["Target", "Model"])